In [ ]:
#!pip install datasets
#!pip install -q -U bitsandbytes transformers peft accelerate datasets
#!pip install -U bitsandbytes>=0.46.1

In [ ]:


import os
import torch
from google.colab import drive
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Setup Google Drive for Permanent Saving
drive.mount('/content/drive')
drive_save_path = "/content/drive/MyDrive/alpaca-portfolio-project"
os.makedirs(drive_save_path, exist_ok=True)

# Load Dataset and Split for Evaluation (Required for Early Stopping)
dataset = load_dataset("tatsu-lab/alpaca")
# We take a small subset for training/eval to speed up learning for a portfolio
split_data = dataset["train"].train_test_split(test_size=0.05, seed=42)
train_data = split_data["train"]
eval_data = split_data["test"]

# Load Model with 4-bit Quantization (QLoRA)
model_name = "EleutherAI/gpt-neo-1.3B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# Prepare for 4-bit training
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# Tokenize (Optimized for Instruction format)
def tokenize_fn(batch):
    prompt = "Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n"
    texts = []
    for i in range(len(batch["instruction"])):
        text = f"{prompt}### Instruction:\n{batch['instruction'][i]}\n"
        if batch["input"][i]:
            text += f"### Input:\n{batch['input'][i]}\n"
        text += f"### Response:\n{batch['output'][i]}"
        texts.append(text)

    return tokenizer(texts, truncation=True, padding="max_length", max_length=256) # 256 is usually enough for Alpaca

tokenized_train = train_data.map(tokenize_fn, batched=True, remove_columns=train_data.column_names)
tokenized_eval = eval_data.map(tokenize_fn, batched=True, remove_columns=eval_data.column_names)

# LoRA Config
lora_config = LoraConfig(
    r=16, # Increased rank for better learning
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)



# Training Arguments with Drive Path & Early Stopping
training_args = TrainingArguments(
    output_dir=drive_save_path,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    eval_strategy="steps",        # <--- Change evaluation_strategy to eval_strategy
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    save_total_limit=2,
    report_to="none"
)

# Trainer with Early Stopping Callback
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] # Stops if loss doesn't improve for 3 eval cycles
)

# Fine-tune
trainer.train()

# Final Save
model.save_pretrained(os.path.join(drive_save_path, "final_model"))
tokenizer.save_pretrained(os.path.join(drive_save_path, "final_model"))
print(f"Project saved to: {drive_save_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/316 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-1.3B
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
transformer.h.{0...23}.attn.attention.masked_bias | UNEXPECTED |  | 
transformer.h.{0...22}.attn.attention.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
100,1.423327,1.368685
200,1.318500,1.339080
300,1.352381,1.325345
400,1.298374,1.316583
500,1.362602,1.307860
600,1.285451,1.303707
700,1.333504,1.299165
800,1.332402,1.295489
900,1.307518,1.291009


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt